# Portfolio risk decomposition, budgeting, allocation, and factor engines

This notebook covers position-level risk analytics and capital allocation exposed by `finstack_quant.portfolio`:

- **Parametric VaR / ES decomposition** -- Euler (component) contributions, plus marginal and incremental VaR, from a covariance matrix.
- **Historical VaR / ES decomposition** -- the same Euler split driven by a scenario P&L matrix.
- **Stress attribution** -- per-position loss attribution across historical tail scenarios.
- **Factor-model engines** -- computed factor stress, position what-if, and credit-vol report helpers.
- **Risk budgeting** -- comparing each position's component VaR against a target risk share and flagging breaches.
- **Capital allocation** -- inverse-volatility and risk-budget weighting schemes.

## Imports

In [1]:
import json
import math
from datetime import date

import pandas as pd

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.factor_model.credit import CreditFactorModel
from finstack_quant.portfolio import (
    CreditVolReport,
    Portfolio,
    RiskDecomposition,
    allocate_weights,
    build_credit_vol_report,
    build_stress_attribution,
    evaluate_risk_budget_typed,
    factor_stress,
    historical_var_decomposition_typed,
    parametric_var_decomposition_typed,
    position_component_var,
    position_what_if,
    validate_allocation_json,
)

POSITIONS = ["Equity", "Credit", "Rates"]
WEIGHTS = [0.50, 0.30, 0.20]
# Annualised return covariance across the three sleeves
COVARIANCE = [
    [0.0400, 0.0120, -0.0020],
    [0.0120, 0.0200, 0.0010],
    [-0.0020, 0.0010, 0.0064],
]

## Parametric VaR / ES decomposition (Euler)

Under the Euler principle the position **component VaRs sum exactly to the portfolio VaR** (the residual is numerically zero). Each contribution also reports:

- **marginal VaR** -- sensitivity of portfolio VaR to the position weight,
- **incremental VaR** -- the change in portfolio VaR from removing the position (when `compute_incremental=True`).

In [2]:
decomp = parametric_var_decomposition_typed(
    POSITIONS, WEIGHTS, COVARIANCE, confidence=0.99, compute_incremental=True
)

print(f"Portfolio VaR (99%): {decomp.portfolio_var:.4f}    ES: {decomp.portfolio_es:.4f}    method: {decomp.method}")
pd.DataFrame([
    {
        "position": c.position_id,
        "component_var": c.component_var,
        "relative_var": c.relative_var,
        "marginal_var": c.marginal_var,
        "incremental_var": c.incremental_var,
    }
    for c in decomp.var_contributions
])

Portfolio VaR (99%): 0.2885    ES: 0.3305    method: parametric


,position,component_var,relative_var,marginal_var,incremental_var
0,Equity,0.217626,0.754422,0.435252,0.179949
1,Credit,0.068665,0.238033,0.228883,0.057513
2,Rates,0.002176,0.007544,0.010881,-0.000225


In [3]:
total_component = sum(c.component_var for c in decomp.var_contributions)
print(f"sum(component VaR) = {total_component:.6f}")
print(f"portfolio VaR      = {decomp.portfolio_var:.6f}")
print(f"Euler residual     = {decomp.euler_residual:.2e}")
print(f"sum(relative VaR)  = {sum(c.relative_var for c in decomp.var_contributions):.6f}")
print(f"component VaR(Equity) via helper = {position_component_var(decomp, 'Equity'):.6f}")

sum(component VaR) = 0.288467
portfolio VaR      = 0.288467
Euler residual     = 5.55e-17
sum(relative VaR)  = 1.000000
component VaR(Equity) via helper = 0.217626


## Historical VaR / ES decomposition

The historical engine takes a **positions x scenarios** P&L matrix (one row of scenario P&Ls per position) and produces the same Euler split empirically. Marginal / incremental fields are not defined for the historical method (they come back as `None`).

In [4]:
def scenario_pnls(n_scenarios=250):
    # Deterministic synthetic P&L paths; the credit sleeve carries a small left tail.
    equity = [0.020 * math.sin(i / 7.0) - 0.001 for i in range(n_scenarios)]
    credit = [(-0.080 if i < 6 else 0.004 * math.cos(i / 5.0) + 0.001) for i in range(n_scenarios)]
    rates = [0.005 * math.cos(i / 11.0) for i in range(n_scenarios)]
    return [equity, credit, rates]

hist = historical_var_decomposition_typed(POSITIONS, scenario_pnls(), confidence=0.95)

print(f"Historical VaR (95%): {hist.portfolio_var:.4f}    ES: {hist.portfolio_es:.4f}    method: {hist.method}")
pd.DataFrame([
    {"position": c.position_id, "component_var": c.component_var, "relative_var": c.relative_var}
    for c in hist.var_contributions
])

Historical VaR (95%): 0.0232    ES: 0.0469    method: historical


,position,component_var,relative_var
0,Equity,0.003303,0.142228
1,Credit,0.020025,0.862389
2,Rates,-0.000107,-0.004616


In [ ]:
from finstack_quant.portfolio import historical_var_decomposition

# The non-typed `historical_var_decomposition` returns a plain dict (JSON-friendly),
# the lightweight counterpart to `historical_var_decomposition_typed`.
hist_dict = historical_var_decomposition(POSITIONS, scenario_pnls(250), confidence=0.95)
print(f"portfolio_var={hist_dict['portfolio_var']:.4f}  portfolio_es={hist_dict['portfolio_es']:.4f}")
for c in hist_dict["contributions"]:
    print(f"  {c['position_id']:8} component_var={c['component_var']:.4f}  pct={c['pct_contribution']:.3f}")

## Risk budgeting

`evaluate_risk_budget_typed` compares each position's **actual component VaR** (here taken from the parametric decomposition above) against a **target share** of portfolio VaR. Utilisation above the threshold (1.10 below) marks a breach.

In [5]:
actual_component_var = [c.component_var for c in decomp.var_contributions]
target_shares = [0.50, 0.30, 0.20]

budget = evaluate_risk_budget_typed(
    POSITIONS, actual_component_var, target_shares, decomp.portfolio_var, 1.10
)

print(f"any breach: {budget.has_breach}    total overbudget: {budget.total_overbudget:.4f}")
pd.DataFrame([
    {
        "position": e.position_id,
        "actual_cVaR": e.actual_component_var,
        "target_cVaR": e.target_component_var,
        "utilization": e.utilization,
        "excess": e.excess,
    }
    for e in budget.positions
])

any breach: True    total overbudget: 0.0734


,position,actual_cVaR,target_cVaR,utilization,excess
0,Equity,0.217626,0.144234,1.508845,0.073393
1,Credit,0.068665,0.086540,0.793444,-0.017875
2,Rates,0.002176,0.057693,0.037721,-0.055517


Equity carries far more than its 50% target share of risk (its high variance and positive correlations concentrate component VaR), so it breaches while Credit and Rates sit under budget.

## Capital allocation schemes

`allocate_weights` takes a JSON spec and returns weights and capital per strategy. Two common schemes:

- **inverse_volatility** -- weight inversely to each strategy's realised volatility,
- **risk_budget** -- solve for weights that hit target risk-contribution shares given a covariance matrix.

`validate_allocation_json` checks a spec before running it.

In [6]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/portfolio_risk_decomposition.json").read_text())

inv_vol_spec = _NOTEBOOK_DATA['inv_vol_spec']
validate_allocation_json(json.dumps(inv_vol_spec))
inv = json.loads(allocate_weights(json.dumps(inv_vol_spec)))
pd.DataFrame(inv["allocations"])

,id,weight,capital,volatility
0,low_vol_carry,0.896583,896582.95,0.001140
1,trend,0.023224,23224.23,0.044017
2,value,0.080193,80192.82,0.012748


In [7]:
rb_spec = {
    "scheme": "risk_budget",
    "total_capital": 1_000_000.0,
    "strategies": [
        {"id": "equity_sleeve", "risk_budget": 0.50},
        {"id": "credit_sleeve", "risk_budget": 0.30},
        {"id": "rates_sleeve", "risk_budget": 0.20},
    ],
    "covariance": COVARIANCE,
    "money_decimal_places": 2,
}
rb = json.loads(allocate_weights(json.dumps(rb_spec)))
pd.DataFrame(rb["allocations"])

,id,weight,capital,risk_contribution
0,equity_sleeve,0.270475,270474.98,0.5
1,credit_sleeve,0.241456,241456.36,0.3
2,rates_sleeve,0.488069,488068.66,0.2


## Factor-model stress, what-if, stress attribution, and credit-vol report

The factor-model engine helpers are now computable from Python. The examples below use a compact one-position portfolio for `factor_stress` and `position_what_if`, reuse the historical scenario matrix for `build_stress_attribution`, and build a credit-vol report from typed `RiskDecomposition` / `CreditFactorModel` inputs.

In [8]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/portfolio_risk_decomposition.json").read_text())

stress_attr = build_stress_attribution(POSITIONS, scenario_pnls(250), confidence=0.95)
display(pd.DataFrame([
    {
        "position": entry.position_id,
        "avg_tail_pnl": entry.avg_tail_pnl,
        "pct_of_tail_loss": entry.pct_of_tail_loss,
        "worst_scenario_pnl": entry.worst_scenario_pnl,
    }
    for entry in stress_attr.position_contributions
]))

mc = MarketContext()
mc.insert(DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75), (10.0, 0.55)],
    day_count="act_365f",
))
portfolio_spec = _NOTEBOOK_DATA['portfolio_spec']
portfolio = Portfolio.from_spec(json.dumps(portfolio_spec))

factor_model_config = _NOTEBOOK_DATA['factor_model_config']
config_json = json.dumps(factor_model_config)

stress = factor_stress(portfolio, mc, config_json, "2025-01-15", [("usd_rates", 25.0)])
what_if = position_what_if(
    portfolio,
    mc,
    config_json,
    "2025-01-15",
    [{"kind": "resize", "position_id": "DEP_LONG", "new_quantity": 0.5}],
)

pd.DataFrame([
    {
        "analysis": "factor_stress +25bp",
        "total_pnl": stress.total_pnl,
        "before_risk": None,
        "after_risk": stress.stressed_decomposition.total_risk,
    },
    {
        "analysis": "position_what_if resize to 0.5x",
        "total_pnl": None,
        "before_risk": what_if.before.total_risk,
        "after_risk": what_if.after.total_risk,
    },
])

,position,avg_tail_pnl,pct_of_tail_loss,worst_scenario_pnl
0,Credit,-0.040426,0.862389,-0.080000
1,Equity,-0.006667,0.142228,-0.020999
2,Rates,0.000216,-0.004616,-0.004840


,analysis,total_pnl,before_risk,after_risk
0,factor_stress +25bp,-1238.612349,NaN,0.245162
1,position_what_if resize to 0.5x,NaN,0.24577,0.061443


In [9]:
credit_model = CreditFactorModel.from_json(json.dumps({
    "schema_version": "finstack_quant.credit_factor_model/1",
    "as_of": "2024-03-29",
    "calibration_window": {"start": "2022-03-29", "end": "2024-03-29"},
    "policy": "globally_off",
    "generic_factor": {"name": "CDX IG", "series_id": "cdx.ig.5y"},
    "hierarchy": {"levels": ["rating", "region"]},
    "config": {
        "factors": [],
        "covariance": {"n": 0, "factor_ids": [], "data": []},
        "matching": {"MappingTable": []},
        "pricing_mode": "delta_based",
    },
    "issuer_betas": [],
    "anchor_state": {"pc": 0.0, "by_level": []},
    "static_correlation": {"factor_ids": [], "data": []},
    "vol_state": {"factors": {}, "idiosyncratic": {}},
    "factor_histories": None,
    "diagnostics": {
        "mode_counts": {},
        "bucket_sizes_per_level": [],
        "fold_ups": [],
        "r_squared_histogram": None,
        "tag_taxonomy": {},
    },
}))
credit_decomp = RiskDecomposition.from_json(json.dumps({
    "total_risk": 1.0,
    "measure": "variance",
    "factor_contributions": [
        {"factor_id": "credit::generic", "absolute_risk": 0.10, "relative_risk": 0.10, "marginal_risk": 0.0},
        {"factor_id": "credit::level0::rating::IG", "absolute_risk": 0.20, "relative_risk": 0.20, "marginal_risk": 0.0},
        {"factor_id": "credit::level1::rating.region::IG.EU", "absolute_risk": 0.30, "relative_risk": 0.30, "marginal_risk": 0.0},
    ],
    "residual_risk": 0.40,
    "position_factor_contributions": [
        {"position_id": "POS1", "factor_id": "credit::generic", "risk_contribution": 0.10},
        {"position_id": "POS1", "factor_id": "credit::level0::rating::IG", "risk_contribution": 0.20},
    ],
    "position_residual_contributions": [],
}))
credit_report = build_credit_vol_report(credit_decomp, credit_model, by_position=True)

pd.DataFrame([
    {"bucket": "generic", "risk": credit_report.generic},
    *[
        {"bucket": level.level_name, "risk": level.total}
        for level in credit_report.by_level
    ],
    {"bucket": "idiosyncratic", "risk": credit_report.idiosyncratic_total},
])

,bucket,risk
0,generic,0.1
1,Rating,0.2
2,Region,0.3
3,idiosyncratic,0.4


## Typed result & enum classes

The risk APIs return strongly-typed result objects (rather than loose dicts). This cell names the concrete types reachable from the objects produced above, and shows the portfolio enum constructors used when authoring optimization/trade specs.

In [ ]:
from finstack_quant.portfolio import (
    PositionRiskDecomposition,
    PositionVarContribution,
    PositionEsContribution,
    RiskBudgetResult,
    PositionBudgetEntry,
    StressAttribution,
    StressPositionEntry,
    StressResult,
    WhatIfResult,
    TailScenarioBreakdown,
    FactorContribution,
    PositionFactorContribution,
    PositionResidualContribution,
    LevelVolContribution,
    PositionVolContribution,
    FactorContributionDelta,
    DecompositionConfig,
    TradeDirection,
    TradeType,
    WeightingScheme,
    OptimizationStatus,
    MissingMetricPolicy,
    Inequality,
    VolHorizon,
)

# Result types reachable from objects produced earlier in this notebook
checks = {
    "PositionRiskDecomposition": isinstance(decomp, PositionRiskDecomposition),
    "PositionVarContribution": isinstance(decomp.var_contributions[0], PositionVarContribution),
    "PositionEsContribution": isinstance(decomp.es_contributions[0], PositionEsContribution),
    "RiskBudgetResult": isinstance(budget, RiskBudgetResult),
    "PositionBudgetEntry": isinstance(budget.positions[0], PositionBudgetEntry),
    "StressAttribution": isinstance(stress_attr, StressAttribution),
    "StressPositionEntry": isinstance(stress_attr.position_contributions[0], StressPositionEntry),
    "TailScenarioBreakdown": all(isinstance(t, TailScenarioBreakdown) for t in stress_attr.tail_scenarios),
    "StressResult": isinstance(stress, StressResult),
    "WhatIfResult": isinstance(what_if, WhatIfResult),
    "FactorContribution": isinstance(credit_decomp.factor_contributions[0], FactorContribution),
    "PositionFactorContribution": isinstance(credit_decomp.position_factor_contributions[0], PositionFactorContribution),
    "PositionResidualContribution": all(isinstance(r, PositionResidualContribution) for r in credit_decomp.position_residual_contributions),
    "LevelVolContribution": all(isinstance(l, LevelVolContribution) for l in credit_report.by_level),
    "PositionVolContribution": all(isinstance(p, PositionVolContribution) for p in (credit_report.by_position or [])),
    "FactorContributionDelta": all(isinstance(d, FactorContributionDelta) for d in what_if.delta),
    "DecompositionConfig": isinstance(DecompositionConfig.parametric_95(), DecompositionConfig),
}
print("Typed result classes confirmed:")
for name, ok in checks.items():
    print(f"  {'OK ' if ok else 'XX '} {name}")

# Portfolio enums (used when building optimization / trade-list / risk specs)
print("\nEnum constructors:")
print("  TradeDirection:", TradeDirection.buy(), TradeDirection.sell(), TradeDirection.hold())
print("  TradeType:", TradeType.new_position(), TradeType.existing(), TradeType.close_out())
print("  WeightingScheme:", WeightingScheme.value_weight(), WeightingScheme.notional_weight())
print("  OptimizationStatus.unbounded():", OptimizationStatus.unbounded())
print("  MissingMetricPolicy:", MissingMetricPolicy.strict(), MissingMetricPolicy.zero(), MissingMetricPolicy.exclude())
print("  Inequality:", Inequality.le(), Inequality.ge(), Inequality.eq())
print("  VolHorizon:", VolHorizon.one_step(), VolHorizon.unconditional())

## Takeaways

- `parametric_var_decomposition_typed` returns an Euler VaR / ES split whose component VaRs sum to the portfolio VaR (residual ~ 0), with optional marginal and incremental VaR.
- `historical_var_decomposition_typed` produces the same split from a positions x scenarios P&L matrix; marginal / incremental are `None` for the historical method.
- `build_stress_attribution` converts scenario P&Ls into per-position tail-loss attribution.
- `factor_stress` and `position_what_if` now compute from `Portfolio`, `MarketContext`, and factor-model config inputs.
- `build_credit_vol_report` summarizes credit-factor risk by generic, hierarchy level, and optionally position.
- `evaluate_risk_budget_typed` flags positions whose component VaR exceeds their target risk share beyond a utilisation threshold.
- `allocate_weights` (with `validate_allocation_json`) sizes strategies under inverse-volatility or risk-budget schemes from a JSON spec.